In [1]:
# --- STEP 1: Load Libraries ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- STEP 2: Load & Clean Dataset ---
# Read CSV with encoding fix in case of special characters
df = pd.read_csv('spam.csv', encoding='latin-1')

# Keep only the essential columns (v1 = target, v2 = message text)
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

# Map text labels to numbers: ham -> 0, spam -> 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

print("Dataset Shape:", df.shape)
print(df.head())

# --- STEP 3: Split into Features (X) and Target (y) ---
X = df['message']
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- STEP 4: Convert Text into Numbers (TF-IDF Vectorization) ---
# TF-IDF measures how important a word is in a message relative to all messages
vectorizer = TfidfVectorizer(stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# --- STEP 5: Train Naive Bayes Classifier ---
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# --- STEP 6: Evaluate Model Performance ---
y_pred = model.predict(X_test_tfidf)

print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# --- STEP 7: Test with Custom Messages ---
sample_messages = [
    "Free msg: Claim your $1000 gift card now! Reply WIN to claim.",
    "Hey mom, I will be home late for dinner today."
]
sample_tfidf = vectorizer.transform(sample_messages)
predictions = model.predict(sample_tfidf)

for msg, pred in zip(sample_messages, predictions):
    print(f"Message: '{msg}' --> Prediction: {'SPAM' if pred == 1 else 'HAM'}")

Dataset Shape: (5572, 3)
  label                                            message  label_num
0   ham  Go until jurong point, crazy.. Available only ...          0
1   ham                      Ok lar... Joking wif u oni...          0
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...          1
3   ham  U dun say so early hor... U c already then say...          0
4   ham  Nah I don't think he goes to usf, he lives aro...          0
Accuracy: 96.68%

Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98       965
           1       1.00      0.75      0.86       150

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115

Message: 'Free msg: Claim your $1000 gift card now! Reply WIN to claim.' --> Prediction: SPAM
Message: 'Hey mom, I will be home late for dinner today.' --> Prediction: HAM
